# INEN BAU uplift — subir 1.A.2 de ~10.36 Mt a ~22 Mt (2050)

**Objetivo.** El BAU actual produce **10.36 MtCO2e** en 1.A.2 (Manufacturing Industries and Construction) en 2050. La SNBC Reference (Figura 27, p.57-58) muestra ~**22 Mt** en 2050. Hay que **subir** la trayectoria moviendo los drivers de INEN/IPPU sin tocar la calibración histórica (tp 0-7). Es el inverso de lo que hicimos en SCOE.

**Diagnóstico.** En `sisepuede_raw_input_morocco_fuels.csv`:

| Driver | tp0-tp7 (calibración) | tp35 (2050) | Problema |
|---|---|---|---|
| `scalar_inen_energy_demand_*` (20 industrias) | 1.000 → 0.996 | **0.890** | Declina cuando SNBC dice que sube |
| `elasticity_ippu_cement_production_to_gdp` | 0.30 | **0.30** | Bajo (SNBC: cemento crece con APC) |
| `elasticity_ippu_chemicals_production_to_gdp` | 0.50 | **0.50** | Bajo (OCP expande fosfatos masivamente) |
| `elasticity_ippu_mining_production_to_gdp` | 0.50 | **0.50** | Bajo (minería de fosfato crece) |
| `elasticity_ippu_metals_production_to_gdp` | 0.50 | **0.50** | Bajo (acero, OCP infrastructure) |

**Estrategia.** Tres palancas, todas sólo desde tp=8:
1. **Uplift multiplicativo** sobre los 20 `scalar_inen_energy_demand_*`: factor que rampa de 1.00 en tp=8 a **1.70** en tp=35. Resultado: scalar pasa de 0.89 → ~1.51 en 2050.
2. **Subir elasticidades de producción** para los sectores que dominan 1.A.2 en Morocco:
   - Cement: 0.30 → **0.40** (límite por SNBC: cemento crece sub-PIB)
   - Chemicals: 0.50 → **0.80** (expansión OCP de ácido fosfórico)
   - Mining: 0.50 → **0.80** (minería de fosfato OCP)
   - Metals: 0.50 → **0.70** (acero + infraestructura OCP)
3. **No tocar `demscalar_ippu_*`** — reservado para shocks específicos (commissioning de plantas).

Esto deja intacta la calibración histórica y enfoca el aumento en los combustibles dominantes de Fig 27 (petroleum coke = cemento; diésel/HFO = minería + manufactura).

**Salida.** Nuevo CSV: `sisepuede_raw_input_morocco_fuels_inen_bau.csv` en `input_data/`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path('/Users/fabianfuentes/git/ssp_morocco')
INPUT_DIR = REPO / 'ssp_modeling' / 'input_data'

INPUT_FILE = INPUT_DIR / 'sisepuede_raw_input_morocco_fuels_scoe.csv'
OUTPUT_FILE = INPUT_DIR / 'sisepuede_raw_input_morocco_fuels_scoe_inen.csv'

# --- Knobs para iterar ---
TP_CALIB_END = 7              # no tocar tp 0..7
TP_TARGET = 35                # tp del objetivo (2050)

# Uplift sobre scalar_inen_energy_demand_* (todos los 20 sectores)
UPLIFT_START = 1.00           # factor en tp=8
UPLIFT_END = 1.70             # factor en tp=35 (= scalar pasa de 0.89 a ~1.51)
UPLIFT_TAIL = 1.70            # constante para tp>35

# Nuevas elasticidades de producción (tp>7)
NEW_ELAS_CEMENT = 0.40        # de 0.30
NEW_ELAS_CHEMICALS = 0.80     # de 0.50 (OCP)
NEW_ELAS_MINING = 0.80        # de 0.50 (fosfato)
NEW_ELAS_METALS = 0.70        # de 0.50

pd.options.display.float_format = '{:.4f}'.format
df = pd.read_csv(INPUT_FILE)
print(f'Loaded {INPUT_FILE.name}: {df.shape}  tp=[{df.time_period.min()}, {df.time_period.max()}]')

Loaded sisepuede_raw_input_morocco_fuels_scoe.csv: (56, 2442)  tp=[0, 55]


## 1 · Auditoría — valores actuales de los drivers

In [2]:
INEN_SCALAR_COLS = sorted([c for c in df.columns if c.startswith('scalar_inen_energy_demand_')])
ELAS_COLS = {
    'elasticity_ippu_cement_production_to_gdp': NEW_ELAS_CEMENT,
    'elasticity_ippu_chemicals_production_to_gdp': NEW_ELAS_CHEMICALS,
    'elasticity_ippu_mining_production_to_gdp': NEW_ELAS_MINING,
    'elasticity_ippu_metals_production_to_gdp': NEW_ELAS_METALS,
}

snapshot_tps = [0, 7, 8, 15, 25, 35, 45, 55]

audit_cols = ['time_period', 'year'] + list(ELAS_COLS.keys()) + INEN_SCALAR_COLS[:3]
print(f'INEN scalars: {len(INEN_SCALAR_COLS)} columnas (mostrando 3 para abreviar)')
df.loc[df.time_period.isin(snapshot_tps), audit_cols].set_index('time_period')

INEN scalars: 20 columnas (mostrando 3 para abreviar)


,year,elasticity_ippu_cement_production_to_gdp,elasticity_ippu_chemicals_production_to_gdp,elasticity_ippu_mining_production_to_gdp,elasticity_ippu_metals_production_to_gdp,scalar_inen_energy_demand_cement,scalar_inen_energy_demand_chemicals,scalar_inen_energy_demand_electronics
time_period,,,,,,,,
0,2015,0.3000,0.5000,0.5000,0.5000,1.0000,1.0000,1.0000
7,2022,0.3000,0.5000,0.5000,0.5000,0.9960,0.9960,0.9960
8,2023,0.3000,0.5000,0.5000,0.5000,0.9920,0.9920,0.9920
15,2030,0.3000,0.5000,0.5000,0.5000,0.9645,0.9645,0.9645
25,2040,0.3000,0.5000,0.5000,0.5000,0.9265,0.9265,0.9265
35,2050,0.3000,0.5000,0.5000,0.5000,0.8901,0.8901,0.8901
45,2060,0.3000,0.5000,0.5000,0.5000,0.8901,0.8901,0.8901
55,2070,0.3000,0.5000,0.5000,0.5000,0.8901,0.8901,0.8901


## 2 · Construir trayectoria de uplift

Rampa lineal de `UPLIFT_START` en tp=8 a `UPLIFT_END` en tp=35; constante en `UPLIFT_TAIL` para tp>35.

In [3]:
tp = df['time_period'].to_numpy()
uplift = np.ones_like(tp, dtype=float)

ramp_mask = (tp > TP_CALIB_END) & (tp <= TP_TARGET)
uplift[ramp_mask] = np.interp(
    tp[ramp_mask],
    [TP_CALIB_END + 1, TP_TARGET],
    [UPLIFT_START, UPLIFT_END],
)
uplift[tp > TP_TARGET] = UPLIFT_TAIL

pd.DataFrame({'time_period': tp, 'year': df['year'], 'uplift_factor': uplift}).query('time_period in @snapshot_tps')

,time_period,year,uplift_factor
0,0,2015,1.0000
7,7,2022,1.0000
8,8,2023,1.0000
15,15,2030,1.1815
25,25,2040,1.4407
35,35,2050,1.7000
45,45,2060,1.7000
55,55,2070,1.7000


## 3 · Aplicar cambios

1. Multiplicar los 20 `scalar_inen_energy_demand_*` por `uplift_factor`.
2. Subir las 4 elasticidades clave de producción para tp > 7.

In [4]:
df_new = df.copy()
forward = df_new['time_period'] > TP_CALIB_END

# Lever 1: uplift INEN energy demand scalars
for col in INEN_SCALAR_COLS:
    df_new[col] = df_new[col].to_numpy() * uplift

# Lever 2: subir elasticidades de producción
for col, new_val in ELAS_COLS.items():
    df_new.loc[forward, col] = new_val

# Snapshot comparison
preview_cols = list(ELAS_COLS.keys()) + ['scalar_inen_energy_demand_cement', 'scalar_inen_energy_demand_chemicals', 'scalar_inen_energy_demand_mining']
print('=== After ===')
df_new.loc[df_new.time_period.isin(snapshot_tps), ['time_period', 'year'] + preview_cols].set_index('time_period')

=== After ===


,year,elasticity_ippu_cement_production_to_gdp,elasticity_ippu_chemicals_production_to_gdp,elasticity_ippu_mining_production_to_gdp,elasticity_ippu_metals_production_to_gdp,scalar_inen_energy_demand_cement,scalar_inen_energy_demand_chemicals,scalar_inen_energy_demand_mining
time_period,,,,,,,,
0,2015,0.3000,0.5000,0.5000,0.5000,1.0000,1.0000,1.0000
7,2022,0.3000,0.5000,0.5000,0.5000,0.9960,0.9960,0.9960
8,2023,0.4000,0.8000,0.8000,0.7000,0.9920,0.9920,0.9920
15,2030,0.4000,0.8000,0.8000,0.7000,1.1395,1.1395,1.1395
25,2040,0.4000,0.8000,0.8000,0.7000,1.3349,1.3349,1.3349
35,2050,0.4000,0.8000,0.8000,0.7000,1.5131,1.5131,1.5131
45,2060,0.4000,0.8000,0.8000,0.7000,1.5131,1.5131,1.5131
55,2070,0.4000,0.8000,0.8000,0.7000,1.5131,1.5131,1.5131


## 4 · Verificación — calibración histórica intacta

In [5]:
modified_cols = INEN_SCALAR_COLS + list(ELAS_COLS.keys())
diff = (df_new[modified_cols] - df[modified_cols]).abs()
print(f'Total columnas modificadas: {len(modified_cols)}')
print(f'Filas con cambio en al menos una columna: {(diff.sum(axis=1) > 1e-12).sum()}')
print()
calib_diff = diff.loc[df['time_period'] <= TP_CALIB_END].max().max()
print(f'Max diff en tp=0..7 (debe ser 0): {calib_diff:.2e}')
assert calib_diff < 1e-12, 'Calibración histórica fue modificada — abortar.'
print('OK: calibración tp=0..7 intacta.')

Total columnas modificadas: 24
Filas con cambio en al menos una columna: 48

Max diff en tp=0..7 (debe ser 0): 0.00e+00
OK: calibración tp=0..7 intacta.


## 5 · Guardar nuevo CSV

In [6]:
df_new.to_csv(OUTPUT_FILE, index=False)
print(f'Wrote {OUTPUT_FILE}')
print(f'Shape: {df_new.shape}')

Wrote /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/input_data/sisepuede_raw_input_morocco_fuels_scoe_inen.csv
Shape: (56, 2442)


## 6 · Próximo paso

Correr el modelo con `sisepuede_raw_input_morocco_fuels_inen_bau.csv` y verificar 1.A.2 en 2050:
- Si **<18 Mt**: subir más → `UPLIFT_END=2.00`, `NEW_ELAS_CHEMICALS=1.00`, `NEW_ELAS_MINING=1.00`.
- Si **>26 Mt**: bajar un poco → `UPLIFT_END=1.50`, `NEW_ELAS_METALS=0.60`.
- Si **20-23 Mt**: 👍 perfecto.

**Notas:**
- Esto también va a subir las emisiones IPPU (Fig 28 SNBC: procesos industriales +80%). Eso es consistente porque las elasticidades de producción mueven *ambos* — INEN (combustión de combustibles) y IPPU (proceso químico).
- Si quieres targetear OCP commissioning (saltos discretos en años específicos), usa `demscalar_ippu_chemicals` con valores >1 en años de entrada en operación de nuevas plantas.
- El petcoke de la cementera domina la barra azul de Fig 27. Si cement no crece lo suficiente, el petcoke se queda corto — vigila eso al validar.